# Machine Learning–Guided Cystatin C Testing

**Working objective:** Determine whether routinely available clinical data can identify adults whose cystatin C–based estimated kidney function is substantially lower than their creatinine-based estimate.

This first notebook:

1. Downloads NHANES 1999–2000 and 2001–2002 data.
2. Merges cystatin C, demographics, and biochemistry files.
3. Standardizes the serum creatinine variable across cycles.
4. Applies initial adult eligibility criteria.
5. Calculates 2021 CKD-EPI creatinine- and cystatin C–based eGFR.
6. Creates discordance and hidden-CKD outcomes.
7. Produces a feasibility summary and saves the analytic dataset.

**Important:** This is a feasibility and data-building notebook. It does not yet fit the final machine-learning models.

In [ ]:
# Cell 1 — Imports and project folders

import io
import os
import sys
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

PROJECT_DIR = Path("/content/cystatin_c_project")
RAW_DIR = PROJECT_DIR / "raw"
CLEAN_DIR = PROJECT_DIR / "clean"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for folder in [PROJECT_DIR, RAW_DIR, CLEAN_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Environment prepared")
print("Project directory:", PROJECT_DIR)

In [ ]:
# Cell 2 — Robust NHANES downloader

def download_xpt(name, url, overwrite=False):
    destination = RAW_DIR / f"{name}.XPT"

    if destination.exists() and not overwrite:
        print(f"↪ Using existing file: {destination.name}")
        return destination

    response = requests.get(url, timeout=180)
    response.raise_for_status()

    content = response.content
    content_type = response.headers.get("content-type", "").lower()

    if "text/html" in content_type or b"<html" in content[:500].lower():
        raise RuntimeError(
            f"CDC returned an HTML page instead of an XPT file for {name}.\nURL: {url}"
        )

    if len(content) < 1000:
        raise RuntimeError(
            f"Downloaded file for {name} is unexpectedly small: {len(content)} bytes"
        )

    destination.write_bytes(content)

    print(
        f"✅ Downloaded {name}: "
        f"{destination.stat().st_size / 1024:.1f} KB"
    )
    return destination


FILE_URLS = {
    # Cystatin C surplus serum files
    "cystatin_1999_2000":
        "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/SSCYST_A.XPT",
    "cystatin_2001_2002":
        "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/SSCYST_B.XPT",

    # Demographics
    "demo_1999_2000":
        "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/DEMO.XPT",
    "demo_2001_2002":
        "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/DEMO_B.XPT",

    # Standard biochemistry
    "biochem_1999_2000":
        "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/LAB18.XPT",
    "biochem_2001_2002":
        "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/L40_B.XPT",
}

downloaded = {
    name: download_xpt(name, url)
    for name, url in FILE_URLS.items()
}

print("\n✅ All required files are available")

In [ ]:
# Cell 3 — Read and standardize XPT files

def read_nhanes_xpt(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_sas(path, format="xport", encoding="utf-8")
    df.columns = df.columns.str.upper().str.strip()

    if "SEQN" not in df.columns:
        raise ValueError(f"SEQN is missing from {path.name}")

    df["SEQN"] = pd.to_numeric(
        df["SEQN"], errors="coerce"
    ).astype("Int64")

    print(f"{path.name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df


cys_1999 = read_nhanes_xpt(downloaded["cystatin_1999_2000"])
cys_2001 = read_nhanes_xpt(downloaded["cystatin_2001_2002"])

demo_1999 = read_nhanes_xpt(downloaded["demo_1999_2000"])
demo_2001 = read_nhanes_xpt(downloaded["demo_2001_2002"])

biochem_1999 = read_nhanes_xpt(downloaded["biochem_1999_2000"])
biochem_2001 = read_nhanes_xpt(downloaded["biochem_2001_2002"])

In [ ]:
# Cell 4 — Validate cystatin C files

def inspect_cystatin_file(df, cycle):
    print(f"\n{'=' * 65}")
    print(cycle)
    print(f"{'=' * 65}")
    print("Rows:", f"{len(df):,}")

    for variable in ["SEQN", "SSCYPC", "WTSCY4YR"]:
        if variable in df.columns:
            print(
                f"✅ {variable}: "
                f"{df[variable].notna().sum():,} nonmissing"
            )
        else:
            print(f"❌ {variable}: not found")

    if "SSCYPC" in df.columns:
        print("\nCystatin C summary:")
        print(pd.to_numeric(df["SSCYPC"], errors="coerce").describe())


inspect_cystatin_file(cys_1999, "NHANES 1999–2000")
inspect_cystatin_file(cys_2001, "NHANES 2001–2002")

In [ ]:
# Cell 5 — Standardize serum creatinine variable names

# NHANES 1999–2000 uses LBXSCR.
# NHANES 2001–2002 uses LBDSCR for conventional units (mg/dL).
# LBDSCRSI is the SI-unit version and should not be used in the adult eGFR equation.

if "LBXSCR" not in biochem_1999.columns:
    raise ValueError(
        "Expected 1999–2000 serum creatinine variable LBXSCR was not found."
    )

if "LBDSCR" in biochem_2001.columns:
    biochem_2001 = biochem_2001.rename(columns={"LBDSCR": "LBXSCR"})
elif "LBXSCR" not in biochem_2001.columns:
    raise ValueError(
        "No conventional-unit serum creatinine variable was found for 2001–2002."
    )

print("✅ Serum creatinine standardized to LBXSCR")
print(
    "1999–2000 nonmissing creatinine:",
    f"{biochem_1999['LBXSCR'].notna().sum():,}"
)
print(
    "2001–2002 nonmissing creatinine:",
    f"{biochem_2001['LBXSCR'].notna().sum():,}"
)

In [ ]:
# Cell 6 — Check required variables

def check_variables(df, file_name, variables):
    print(f"\n{'=' * 65}")
    print(file_name)
    print(f"{'=' * 65}")

    for variable in variables:
        if variable in df.columns:
            print(
                f"✅ {variable}: "
                f"{df[variable].notna().sum():,} nonmissing"
            )
        else:
            print(f"⚠️ {variable}: not present")


check_variables(
    demo_1999,
    "1999–2000 demographics",
    ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH1",
     "RIDEXPRG", "SDMVPSU", "SDMVSTRA"]
)

check_variables(
    demo_2001,
    "2001–2002 demographics",
    ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH1",
     "RIDEXPRG", "SDMVPSU", "SDMVSTRA"]
)

check_variables(
    biochem_1999,
    "1999–2000 biochemistry",
    ["SEQN", "LBXSCR", "LBXSBU", "LBXSAL", "LBXSGL"]
)

check_variables(
    biochem_2001,
    "2001–2002 biochemistry",
    ["SEQN", "LBXSCR", "LBXSBU", "LBXSAL", "LBXSGL"]
)

In [ ]:
# Cell 7 — Merge each cycle by SEQN

def assert_unique_seqn(df, name):
    duplicates = df["SEQN"].duplicated().sum()
    if duplicates:
        raise ValueError(f"{name} contains {duplicates} duplicate SEQN values.")


for name, df in [
    ("cys_1999", cys_1999),
    ("cys_2001", cys_2001),
    ("demo_1999", demo_1999),
    ("demo_2001", demo_2001),
    ("biochem_1999", biochem_1999),
    ("biochem_2001", biochem_2001),
]:
    assert_unique_seqn(df, name)


merged_1999 = (
    cys_1999
    .merge(demo_1999, on="SEQN", how="left", validate="one_to_one")
    .merge(biochem_1999, on="SEQN", how="left", validate="one_to_one")
)

merged_2001 = (
    cys_2001
    .merge(demo_2001, on="SEQN", how="left", validate="one_to_one")
    .merge(biochem_2001, on="SEQN", how="left", validate="one_to_one")
)

merged_1999["CYCLE"] = "1999-2000"
merged_2001["CYCLE"] = "2001-2002"

print("✅ Merges completed")
print("1999–2000:", merged_1999.shape)
print("2001–2002:", merged_2001.shape)

In [ ]:
# Cell 8 — Combine cycles and prepare numeric variables

combined = pd.concat(
    [merged_1999, merged_2001],
    ignore_index=True,
    sort=False
)

numeric_columns = [
    "RIDAGEYR", "RIAGENDR", "RIDRETH1", "RIDEXPRG",
    "SSCYPC", "LBXSCR", "WTSCY4YR",
    "SDMVPSU", "SDMVSTRA",
    "LBXSBU", "LBXSAL", "LBXSGL"
]

for column in numeric_columns:
    if column in combined.columns:
        combined[column] = pd.to_numeric(
            combined[column], errors="coerce"
        )

print("Combined rows:", f"{len(combined):,}")
print("Combined columns:", f"{combined.shape[1]:,}")

In [ ]:
# Cell 9 — Apply initial eligibility criteria

# NHANES sex coding: 1 = male, 2 = female.
# RIDEXPRG commonly uses 1 = pregnant, 2 = not pregnant, 3 = unknown.
# Males and women without a pregnancy value are not excluded solely because RIDEXPRG is missing.

combined["FEMALE"] = (combined["RIAGENDR"] == 2).astype("Int64")

adult_mask = combined["RIDAGEYR"] >= 20

pregnant_mask = (
    (combined["RIAGENDR"] == 2)
    & (combined["RIDEXPRG"] == 1)
)

valid_measurements_mask = (
    combined["SSCYPC"].notna()
    & combined["LBXSCR"].notna()
    & (combined["SSCYPC"] > 0)
    & (combined["LBXSCR"] > 0)
)

eligible = combined.loc[
    adult_mask
    & ~pregnant_mask
    & valid_measurements_mask
].copy()

print("=" * 65)
print("INITIAL ANALYTIC SAMPLE")
print("=" * 65)
print("Total cystatin C file rows:", f"{len(combined):,}")
print("Adults age ≥20:", f"{adult_mask.sum():,}")
print(
    "Adults with nonmissing cystatin C:",
    f"{(adult_mask & combined['SSCYPC'].notna()).sum():,}"
)
print(
    "Adults with nonmissing creatinine:",
    f"{(adult_mask & combined['LBXSCR'].notna()).sum():,}"
)
print("Pregnant adults excluded:", f"{(adult_mask & pregnant_mask).sum():,}")
print("Final initial eligible sample:", f"{len(eligible):,}")

print("\nEligible sample by cycle:")
print(eligible["CYCLE"].value_counts().sort_index())

In [ ]:
# Cell 10 — 2021 CKD-EPI adult eGFR equations

def calculate_egfr_creatinine_2021(creatinine, age, female):
    creatinine = pd.to_numeric(creatinine, errors="coerce")
    age = pd.to_numeric(age, errors="coerce")
    female = pd.Series(female, index=creatinine.index).fillna(0).astype(bool)

    kappa = np.where(female, 0.7, 0.9)
    alpha = np.where(female, -0.241, -0.302)
    sex_factor = np.where(female, 1.012, 1.0)

    ratio = creatinine / kappa

    egfr = (
        142
        * np.minimum(ratio, 1) ** alpha
        * np.maximum(ratio, 1) ** -1.200
        * 0.9938 ** age
        * sex_factor
    )

    invalid = (
        creatinine.isna()
        | age.isna()
        | (creatinine <= 0)
        | (age < 18)
    )

    return pd.Series(
        np.where(invalid, np.nan, egfr),
        index=creatinine.index
    )


def calculate_egfr_cystatin_2021(cystatin_c, age, female):
    cystatin_c = pd.to_numeric(cystatin_c, errors="coerce")
    age = pd.to_numeric(age, errors="coerce")
    female = pd.Series(female, index=cystatin_c.index).fillna(0).astype(bool)

    ratio = cystatin_c / 0.8
    sex_factor = np.where(female, 0.932, 1.0)

    egfr = (
        133
        * np.minimum(ratio, 1) ** -0.499
        * np.maximum(ratio, 1) ** -1.328
        * 0.996 ** age
        * sex_factor
    )

    invalid = (
        cystatin_c.isna()
        | age.isna()
        | (cystatin_c <= 0)
        | (age < 18)
    )

    return pd.Series(
        np.where(invalid, np.nan, egfr),
        index=cystatin_c.index
    )


eligible["EGFR_CR"] = calculate_egfr_creatinine_2021(
    eligible["LBXSCR"],
    eligible["RIDAGEYR"],
    eligible["FEMALE"]
)

eligible["EGFR_CYS"] = calculate_egfr_cystatin_2021(
    eligible["SSCYPC"],
    eligible["RIDAGEYR"],
    eligible["FEMALE"]
)

print("✅ eGFR values calculated")
print(
    eligible[["LBXSCR", "SSCYPC", "EGFR_CR", "EGFR_CYS"]]
    .describe()
    .round(2)
)

In [ ]:
# Cell 11 — Create study outcomes

eligible["EGFR_RATIO_CYS_TO_CR"] = (
    eligible["EGFR_CYS"] / eligible["EGFR_CR"]
)

eligible["EGFR_DIFFERENCE_CYS_MINUS_CR"] = (
    eligible["EGFR_CYS"] - eligible["EGFR_CR"]
)

# Primary candidate outcome:
# cystatin C eGFR is more than 30% lower than creatinine eGFR.
eligible["DISCORDANCE_30"] = (
    eligible["EGFR_CYS"] < 0.70 * eligible["EGFR_CR"]
).astype(int)

# Secondary outcome:
# creatinine eGFR appears preserved, but cystatin C eGFR is <60.
eligible["HIDDEN_CKD"] = (
    (eligible["EGFR_CR"] >= 60)
    & (eligible["EGFR_CYS"] < 60)
).astype(int)

# Additional thresholds for sensitivity analyses.
eligible["DISCORDANCE_20"] = (
    eligible["EGFR_CYS"] < 0.80 * eligible["EGFR_CR"]
).astype(int)

eligible["DISCORDANCE_40"] = (
    eligible["EGFR_CYS"] < 0.60 * eligible["EGFR_CR"]
).astype(int)

print("✅ Outcomes created")

In [ ]:
# Cell 12 — Unweighted feasibility results

def summarize_binary_outcome(df, column):
    n = df[column].notna().sum()
    cases = int(df[column].sum())
    prevalence = 100 * cases / n if n else np.nan

    return {
        "Outcome": column,
        "Eligible N": n,
        "Cases": cases,
        "Prevalence (%)": prevalence
    }


feasibility = pd.DataFrame([
    summarize_binary_outcome(eligible, "DISCORDANCE_20"),
    summarize_binary_outcome(eligible, "DISCORDANCE_30"),
    summarize_binary_outcome(eligible, "DISCORDANCE_40"),
    summarize_binary_outcome(eligible, "HIDDEN_CKD"),
])

print("=" * 65)
print("UNWEIGHTED FEASIBILITY RESULTS")
print("=" * 65)
display(feasibility.round({"Prevalence (%)": 2}))

print("\nPrimary outcome by cycle:")
primary_by_cycle = (
    eligible.groupby("CYCLE")["DISCORDANCE_30"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "Eligible N",
        "sum": "Cases",
        "mean": "Prevalence"
    })
)
primary_by_cycle["Prevalence (%)"] = (
    100 * primary_by_cycle.pop("Prevalence")
)
display(primary_by_cycle.round(2))

In [ ]:
# Cell 13 — Weighted prevalence using the special cystatin C weight

# This computes a basic weighted prevalence estimate.
# Final inferential analyses should additionally account for strata and PSU
# using a survey-analysis package or R's survey package.

def weighted_prevalence(df, outcome, weight="WTSCY4YR"):
    analysis = df[[outcome, weight]].dropna().copy()
    analysis = analysis.loc[analysis[weight] > 0]

    if analysis.empty:
        return np.nan

    return np.average(
        analysis[outcome],
        weights=analysis[weight]
    )


weighted_results = []

for outcome in [
    "DISCORDANCE_20",
    "DISCORDANCE_30",
    "DISCORDANCE_40",
    "HIDDEN_CKD"
]:
    weighted_results.append({
        "Outcome": outcome,
        "Weighted prevalence (%)":
            100 * weighted_prevalence(eligible, outcome)
    })

weighted_results = pd.DataFrame(weighted_results)

print("=" * 65)
print("WEIGHTED PREVALENCE ESTIMATES")
print("=" * 65)
display(weighted_results.round(2))

In [ ]:
# Cell 14 — Basic distribution plots

plt.figure(figsize=(8, 5))
plt.hist(
    eligible["EGFR_RATIO_CYS_TO_CR"].dropna(),
    bins=50
)
plt.axvline(0.70, linestyle="--")
plt.xlabel("eGFR cystatin C / eGFR creatinine")
plt.ylabel("Number of participants")
plt.title("Distribution of eGFR Discordance Ratio")
plt.tight_layout()
plt.show()


plt.figure(figsize=(7, 6))
plt.scatter(
    eligible["EGFR_CR"],
    eligible["EGFR_CYS"],
    alpha=0.35
)
max_axis = np.nanpercentile(
    eligible[["EGFR_CR", "EGFR_CYS"]].values,
    99
)
plt.plot([0, max_axis], [0, max_axis], linestyle="--")
plt.xlim(0, max_axis)
plt.ylim(0, max_axis)
plt.xlabel("Creatinine-based eGFR")
plt.ylabel("Cystatin C-based eGFR")
plt.title("Creatinine- vs Cystatin C–Based eGFR")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15 — Save clean data and feasibility tables

analytic_csv = CLEAN_DIR / "nhanes_cystatin_creatinine_analytic.csv"
feasibility_csv = OUTPUT_DIR / "feasibility_unweighted.csv"
weighted_csv = OUTPUT_DIR / "feasibility_weighted.csv"

eligible.to_csv(analytic_csv, index=False)
feasibility.to_csv(feasibility_csv, index=False)
weighted_results.to_csv(weighted_csv, index=False)

print("✅ Saved:", analytic_csv)
print("✅ Saved:", feasibility_csv)
print("✅ Saved:", weighted_csv)

In [ ]:
# Cell 16 — Create a downloadable ZIP of outputs

zip_path = Path("/content/Cystatin_C_Feasibility_Outputs.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in [
        analytic_csv,
        feasibility_csv,
        weighted_csv
    ]:
        zf.write(file_path, arcname=file_path.name)

print("✅ Output ZIP created:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print(
        "Automatic download is only available in Google Colab. "
        "The ZIP remains saved at:",
        zip_path
    )

## What to send back after running the notebook

Send a screenshot or copy the output from:

- **Cell 9:** Final initial eligible sample
- **Cell 12:** `DISCORDANCE_30` cases and prevalence
- **Cell 13:** Weighted prevalence

Those results determine whether the primary 30% discordance outcome has enough cases for machine learning or whether the threshold/design should be adjusted.